# DeepCoder 1.5B baseline / NGU LCBv5 difficulty-split summary

Pulls the `eval_deepcoder_1_5b_*` standalone checkpoint-eval runs from W&B project
`ai2-llm/open_instruct_internal` for the DeepCoder 1.5B baseline N/K sweep (`N=8, K=16`,
`N=4, K=32`, `N=2, K=64`) and the NGU `p` sweep (`p=0.5`, `p=0.75`, `p=0.875`, all at
`N=8, K=16`), 3 seeds each (18 runs total), and reproduces the same
`improvement over initial_eval, by difficulty bucket` summary table as the end of
`deepscaler_ngu_plots.ipynb` -- just the table, no bar charts/line plots.

**Run shape is different from the DeepScaler notebooks.** These training runs don't log
their own eval history point-by-point in one wandb run; instead, each training checkpoint
gets its own **standalone** `eval_deepcoder_1_5b_<setting>_seed<N>[_stepXXX/_remK]` eval-only
wandb run (oe-eval launched separately per checkpoint), so a given (setting, seed) is scattered
across several wandb run ids, one per evaluated step (and occasional failed/retried launches
with a `_rem`/`_remK` suffix). "Best lcbv5 eval over training" is therefore computed by
discovering all matching runs, grouping by (setting, seed), and taking the run with the max
`eval/pass_at_1/code_stdio_lcbv5` -- the run-selection cell below prints exactly which run id
won each group, so it's auditable rather than hardcoded like the DeepScaler notebooks' `RUN_SPECS`.

**Difficulty buckets** come from the initial-model eval run
[`f6nzzvl2`](https://wandb.ai/ai2-llm/open_instruct_internal/runs/f6nzzvl2)
(`DeepSeek-R1-Distill-Qwen-1.5B`, zero-shot, `eval_pass_at_k=32`), using the same
hard/medium/easy split algorithm as the DeepScaler notebooks (hard = solve_rate 0,
remainder split evenly into easy/medium by rank). This replaces an earlier attempt using
[`q9gelapy`](https://wandb.ai/ai2-llm/open_instruct_internal/runs/q9gelapy), which only
had `eval_pass_at_k=1` (a single decode per prompt) -- with a binary 0/1 solve_rate, the
easy/medium split there was an arbitrary partition of ties at `solve_rate=1.0`, and every
trained checkpoint showed a huge, spurious "degradation" against that tautological 100%
initial baseline. `f6nzzvl2`'s 32-sample solve rates are a genuine continuous difficulty
signal, like DeepScaler's dedicated 128-sample difficulty run.


In [1]:
from __future__ import annotations

import json
import re
import tempfile
from pathlib import Path

import pandas as pd
import wandb
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


In [2]:
PROJECT_PATH = "ai2-llm/open_instruct_internal"
LCBV5_EVAL_KEY = "eval/pass_at_1/code_stdio_lcbv5"
# mnoukhov/deepcoder_lcbv5_test has 175 rows and is always the *first* dataset in these runs'
# eval mixers (training checkpoints and f6nzzvl2 both mix it with humanevalplus_test) --
# dataset_index < 175 is therefore the lcbv5-only slice in every table here, no problem_idx
# remapping needed (unlike the DeepScaler AIME/BRUMO notebooks, whose two eval mixers disagreed
# on dataset order).
LCBV5_TABLE_SIZE = 175
DIFFICULTY_RUN_ID = "f6nzzvl2"  # zero-shot DeepSeek-R1-Distill-Qwen-1.5B eval, pass@32, lcbv5+humanevalplus mixer
DIFFICULTY_ORDER = ["hard", "medium", "easy"]

SETTING_LABELS = {
    "baseline": "N=8, K=16",
    "n4_k32": "N=4, K=32",
    "n2_k64": "N=2, K=64",
    "ngu05": "NGU p=0.5",
    "ngu075": "NGU p=0.75",
    "ngu0875": "NGU p=0.875",
}
NAME_RE = re.compile(r"^eval_deepcoder_1_5b_(baseline|n4_k32|n2_k64|ngu05|ngu075|ngu0875)_seed(\d+)")

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
OUTPUT_DIR = NOTEBOOK_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WANDB_CACHE_DIR = NOTEBOOK_DIR / ".wandb_cache"
WANDB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

api = wandb.Api(timeout=90)


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/mnoukhov/.netrc.


## Run discovery: pick each (setting, seed)'s best checkpoint eval

Fetches every `eval_deepcoder_1_5b_*` run, keeps the ones matching
`eval_deepcoder_1_5b_{baseline,ngu05,ngu075,ngu0875}_seed{N}` (ignoring any `_stepXXX`/`_rem`/
`_remK` suffix after the seed number -- those are just retry/resume naming, not a different
setting), and for each (setting, seed) group keeps the run with the highest
`eval/pass_at_1/code_stdio_lcbv5` among `state == "finished"` runs.


In [3]:
runs = api.runs(PROJECT_PATH, filters={"display_name": {"$regex": "^eval_deepcoder_1_5b"}})

candidates: dict[tuple[str, int], list[tuple[float, str, object, str]]] = {}
for r in runs:
    m = NAME_RE.match(r.name)
    if m is None:
        continue
    score = r.summary.get(LCBV5_EVAL_KEY)
    if r.state != "finished" or score is None:
        continue
    key = (m.group(1), int(m.group(2)))
    candidates.setdefault(key, []).append((float(score), r.id, r.summary.get("eval_step"), r.name))

RUN_REGISTRY = []
for (setting_key, seed), options in sorted(candidates.items()):
    best_score, best_run_id, best_eval_step, best_name = max(options, key=lambda o: o[0])
    RUN_REGISTRY.append({
        "setting": SETTING_LABELS[setting_key], "seed": seed, "run_id": best_run_id,
        "eval_step": best_eval_step, "lcbv5": best_score, "n_candidates": len(options),
    })
    print(f"{setting_key:8s} seed{seed}  -> best {best_run_id} (eval_step={best_eval_step}, "
          f"lcbv5={best_score:.3f}, {len(options)} candidate checkpoint(s))")

missing = [(s, seed) for s in SETTING_LABELS for seed in (1, 2, 3) if (s, seed) not in candidates]
if missing:
    print("\nMISSING (setting, seed) groups (no finished eval found):", missing)
else:
    print(f"\nAll {len(SETTING_LABELS)} settings x 3 seeds accounted for ({len(RUN_REGISTRY)} runs registered).")


baseline seed1  -> best 3l7dmkko (eval_step=500, lcbv5=0.183, 1 candidate checkpoint(s))
baseline seed2  -> best 53t8fk7u (eval_step=100, lcbv5=0.171, 5 candidate checkpoint(s))
baseline seed3  -> best fbx3fbon (eval_step=400, lcbv5=0.189, 4 candidate checkpoint(s))
n2_k64   seed1  -> best 03iusfh3 (eval_step=500, lcbv5=0.143, 1 candidate checkpoint(s))
n2_k64   seed2  -> best qfux94k6 (eval_step=200, lcbv5=0.183, 5 candidate checkpoint(s))
n2_k64   seed3  -> best 6609f9u6 (eval_step=500, lcbv5=0.189, 5 candidate checkpoint(s))
n4_k32   seed1  -> best mnxklwfg (eval_step=500, lcbv5=0.160, 1 candidate checkpoint(s))
n4_k32   seed2  -> best 9rwzfz8y (eval_step=200, lcbv5=0.206, 4 candidate checkpoint(s))
n4_k32   seed3  -> best h7l13sdf (eval_step=400, lcbv5=0.194, 5 candidate checkpoint(s))
ngu05    seed1  -> best oelwuijn (eval_step=500, lcbv5=0.143, 1 candidate checkpoint(s))
ngu05    seed2  -> best zirsnpr5 (eval_step=100, lcbv5=0.194, 5 candidate checkpoint(s))
ngu05    seed3  -> be

## Solve-rate-by-index tables (lcbv5-only slice)


In [4]:
def table_path_from_entry(entry) -> str:
    return entry["path"] if isinstance(entry, dict) else entry.path


def download_solve_rate_table(run_id: str, *, table_entry) -> pd.DataFrame:
    # Cached by the resolved table file path -- a specific logged step's table is immutable
    # once written, so this is always safe to cache indefinitely (same convention as
    # deepscaler_ngu_plots.ipynb).
    table_path = table_path_from_entry(table_entry)
    cache_path = WANDB_CACHE_DIR / (table_path.replace("/", "__") + ".csv")
    if cache_path.exists():
        return pd.read_csv(cache_path)
    run = api.run(f"{PROJECT_PATH}/{run_id}")
    with tempfile.TemporaryDirectory() as tmp_dir:
        downloaded = run.file(table_path).download(root=tmp_dir, replace=True)
        with open(downloaded.name) as fh:
            payload = json.load(fh)
    table_df = pd.DataFrame(payload["data"], columns=payload["columns"])
    table_df["dataset_index"] = pd.to_numeric(table_df["dataset_index"], errors="raise").astype(int)
    table_df["solve_rate"] = pd.to_numeric(table_df["solve_rate"], errors="raise").astype(float)
    table_df = table_df.sort_values("dataset_index").reset_index(drop=True)
    table_df.to_csv(cache_path, index=False)
    return table_df


def lcbv5_only(table_df: pd.DataFrame) -> pd.DataFrame:
    return table_df[table_df["dataset_index"] < LCBV5_TABLE_SIZE].reset_index(drop=True)


## Difficulty buckets (from `f6nzzvl2`, zero-shot initial model, pass@32)


In [5]:
difficulty_run = api.run(f"{PROJECT_PATH}/{DIFFICULTY_RUN_ID}")
initial_solve_df = lcbv5_only(download_solve_rate_table(
    DIFFICULTY_RUN_ID, table_entry=difficulty_run.summary["eval/prompt_solve_rate_by_index_table"]
))


def assign_difficulty_buckets(df: pd.DataFrame) -> pd.DataFrame:
    # hard: solve_rate == 0; remainder split evenly into easy (top half) / medium -- same
    # algorithm as deepscaler_ngu_plots.ipynb.
    df = df.copy()
    hard_mask = df["solve_rate"] <= 0
    df.loc[hard_mask, "difficulty"] = "hard"

    remainder = df.loc[~hard_mask].sort_values("solve_rate", ascending=False)
    n_easy = len(remainder) // 2 + (len(remainder) % 2)
    df.loc[remainder.index[:n_easy], "difficulty"] = "easy"
    df.loc[remainder.index[n_easy:], "difficulty"] = "medium"
    df["difficulty"] = pd.Categorical(df["difficulty"], categories=DIFFICULTY_ORDER, ordered=True)
    return df


difficulty_assignments_df = assign_difficulty_buckets(initial_solve_df)
DIFF_LOOKUP = difficulty_assignments_df.set_index("dataset_index")["difficulty"]
INITIAL_BUCKET_PASS_AT_1 = difficulty_assignments_df.groupby("difficulty", observed=True)["solve_rate"].mean()
INITIAL_OVERALL_PASS_AT_1 = initial_solve_df["solve_rate"].mean()  # all 175 lcbv5 prompts, not bucketed -- for the "Total" column

print("lcbv5 difficulty bucket sizes:", difficulty_assignments_df["difficulty"].value_counts().reindex(DIFFICULTY_ORDER).to_dict())
print("initial_eval pass@1 by bucket:", INITIAL_BUCKET_PASS_AT_1.reindex(DIFFICULTY_ORDER).round(3).to_dict())
print("initial_eval overall pass@1:", round(INITIAL_OVERALL_PASS_AT_1, 3))


lcbv5 difficulty bucket sizes: {'hard': 96, 'medium': 39, 'easy': 40}
initial_eval pass@1 by bucket: {'hard': 0.0, 'medium': 0.083, 'easy': 0.679}
initial_eval overall pass@1: 0.174


## Summary table: lcbv5 pass@1 improvement over `initial_eval`, by difficulty bucket

Same table format as the end of `deepscaler_ngu_plots.ipynb`: for each run's own best-over-
training checkpoint, mean solve_rate within each difficulty bucket minus `initial_eval`'s
pass@1 in that bucket (percentage points), then mean ± std across the 3 seeds. Columns read
easy -> hard, left to right.


In [6]:
IMPROVEMENT_TABLE_COLUMN_ORDER = ["easy", "medium", "hard"]
IMPROVEMENT_TABLE_SECTIONS = [
    ("GRPO Baseline", ["N=8, K=16", "N=4, K=32", "N=2, K=64"]),
    ("NGU", ["NGU p=0.5", "NGU p=0.75", "NGU p=0.875"]),
]

per_seed_rows = []
for spec in RUN_REGISTRY:
    run = api.run(f"{PROJECT_PATH}/{spec['run_id']}")
    table_df = lcbv5_only(download_solve_rate_table(
        spec["run_id"], table_entry=run.summary["eval/prompt_solve_rate_by_index_table"]
    ))
    table_df["difficulty"] = table_df["dataset_index"].map(DIFF_LOOKUP)
    per_bucket = table_df.groupby("difficulty", observed=True)["solve_rate"].mean()
    for difficulty in DIFFICULTY_ORDER:
        per_seed_rows.append({
            "setting": spec["setting"], "seed": spec["seed"], "difficulty": difficulty,
            "improvement": (per_bucket[difficulty] - INITIAL_BUCKET_PASS_AT_1[difficulty]) * 100,
        })
    # "total": overall pass@1 across all 175 lcbv5 prompts (not bucketed) -- spec["lcbv5"] is the
    # same wandb-logged eval/pass_at_1/code_stdio_lcbv5 value used to pick this run as the seed's
    # best checkpoint, so it's already exactly this table's overall mean solve_rate.
    per_seed_rows.append({
        "setting": spec["setting"], "seed": spec["seed"], "difficulty": "total",
        "improvement": (spec["lcbv5"] - INITIAL_OVERALL_PASS_AT_1) * 100,
    })

per_seed_df = pd.DataFrame(per_seed_rows)

IMPROVEMENT_TABLE_COLUMNS = [*IMPROVEMENT_TABLE_COLUMN_ORDER, "total"]  # Easy, Medium, Hard, Total


def build_improvement_summary_table() -> pd.DataFrame:
    rows = []
    for group_name, settings in IMPROVEMENT_TABLE_SECTIONS:
        for setting in settings:
            sub = per_seed_df[per_seed_df["setting"] == setting]
            row = {"Group": group_name, "Setting": setting}
            for column in IMPROVEMENT_TABLE_COLUMNS:
                values = sub.loc[sub["difficulty"] == column, "improvement"]
                mean, std, n = values.mean(), values.std(ddof=1), len(values)
                row[column.capitalize()] = f"{mean:.1f} ± {std:.1f}" if n > 1 else f"{mean:.1f}"
            rows.append(row)
    return pd.DataFrame(rows).set_index(["Group", "Setting"])


improvement_summary_df = build_improvement_summary_table()
improvement_summary_df.to_csv(OUTPUT_DIR / "deepcoder_1_5b_improvement_by_difficulty_summary_lcbv5.csv")
print("lcbv5 pass@1 (%) improvement over initial_eval vs. difficulty bucket, mean ± std across seeds")
display(improvement_summary_df)


lcbv5 pass@1 (%) improvement over initial_eval vs. difficulty bucket, mean ± std across seeds


Easy     Medium       Hard       Total
Group         Setting                                                   
GRPO Baseline N=8, K=16     -0.4 ± 2.5  3.7 ± 5.3  0.0 ± 0.0   0.7 ± 0.9
              N=4, K=32      2.9 ± 5.2  2.0 ± 5.1  0.3 ± 0.6   1.3 ± 2.4
              N=2, K=64     -4.6 ± 9.5  2.0 ± 0.0  0.7 ± 0.6  -0.2 ± 2.5
NGU           NGU p=0.5    -1.2 ± 14.6  3.7 ± 3.9  0.7 ± 1.2   0.9 ± 3.6
              NGU p=0.75    -0.4 ± 6.6  5.4 ± 7.4  0.0 ± 0.0   1.1 ± 0.3
              NGU p=0.875   -4.6 ± 3.8  4.6 ± 6.8  1.4 ± 0.6   0.7 ± 1.8

Copyable Markdown source for the table above (printed as plain text so it can be
selected/copied directly, e.g. for pasting into a GitHub PR or `research.md`):


In [7]:
def df_to_markdown(df: pd.DataFrame) -> str:
    # Plain pandas/str implementation (no `tabulate` dependency) so this works in any kernel.
    header = list(df.columns)
    rows = df.astype(str).values.tolist()
    widths = [max(len(h), *(len(r[i]) for r in rows)) if rows else len(h) for i, h in enumerate(header)]
    def fmt_row(cells: list[str]) -> str:
        return "| " + " | ".join(c.ljust(w) for c, w in zip(cells, widths)) + " |"
    lines = [fmt_row(header), fmt_row(["-" * w for w in widths])]
    lines += [fmt_row(r) for r in rows]
    return "\n".join(lines)


print(df_to_markdown(improvement_summary_df.reset_index()))


| Group         | Setting     | Easy        | Medium    | Hard      | Total      |
| ------------- | ----------- | ----------- | --------- | --------- | ---------- |
| GRPO Baseline | N=8, K=16   | -0.4 ± 2.5  | 3.7 ± 5.3 | 0.0 ± 0.0 | 0.7 ± 0.9  |
| GRPO Baseline | N=4, K=32   | 2.9 ± 5.2   | 2.0 ± 5.1 | 0.3 ± 0.6 | 1.3 ± 2.4  |
| GRPO Baseline | N=2, K=64   | -4.6 ± 9.5  | 2.0 ± 0.0 | 0.7 ± 0.6 | -0.2 ± 2.5 |
| NGU           | NGU p=0.5   | -1.2 ± 14.6 | 3.7 ± 3.9 | 0.7 ± 1.2 | 0.9 ± 3.6  |
| NGU           | NGU p=0.75  | -0.4 ± 6.6  | 5.4 ± 7.4 | 0.0 ± 0.0 | 1.1 ± 0.3  |
| NGU           | NGU p=0.875 | -4.6 ± 3.8  | 4.6 ± 6.8 | 1.4 ± 0.6 | 0.7 ± 1.8  |


## Summary table: lcbv5 raw pass@1, best checkpoint per seed

Not an improvement-over-initial number -- just each seed's own best-over-training
`eval/pass_at_1/code_stdio_lcbv5` score (%) from the run registry above, one column per
seed, plus a `Total` column with the mean ± std across the 3 seeds.


In [8]:
def build_raw_score_table() -> pd.DataFrame:
    rows = []
    for group_name, settings in IMPROVEMENT_TABLE_SECTIONS:
        for setting in settings:
            specs = sorted((s for s in RUN_REGISTRY if s["setting"] == setting), key=lambda s: s["seed"])
            row = {"Group": group_name, "Setting": setting}
            scores = []
            for spec in specs:
                pct = spec["lcbv5"] * 100
                scores.append(pct)
                row[f"Seed {spec['seed']}"] = f"{pct:.1f}"
            mean, std = pd.Series(scores).mean(), pd.Series(scores).std(ddof=1)
            row["Total"] = f"{mean:.1f} ± {std:.1f}"
            rows.append(row)

    df = pd.DataFrame(rows)
    seed_cols = sorted((c for c in df.columns if c.startswith("Seed ")), key=lambda c: int(c.split(" ")[1]))
    return df[["Group", "Setting", *seed_cols, "Total"]].set_index(["Group", "Setting"])


raw_score_summary_df = build_raw_score_table()
raw_score_summary_df.to_csv(OUTPUT_DIR / "deepcoder_1_5b_lcbv5_raw_score_summary.csv")
print("lcbv5 pass@1 (%), best checkpoint per seed")
display(raw_score_summary_df)


lcbv5 pass@1 (%), best checkpoint per seed


Seed 1 Seed 2 Seed 3       Total
Group         Setting                                     
GRPO Baseline N=8, K=16     18.3   17.1   18.9  18.1 ± 0.9
              N=4, K=32     16.0   20.6   19.4  18.7 ± 2.4
              N=2, K=64     14.3   18.3   18.9  17.1 ± 2.5
NGU           NGU p=0.5     14.3   19.4   21.1  18.3 ± 3.6
              NGU p=0.75    18.9   18.3   18.3  18.5 ± 0.3
              NGU p=0.875   16.0   19.4   18.9  18.1 ± 1.8

Copyable Markdown source for the table above:


In [9]:
print(df_to_markdown(raw_score_summary_df.reset_index()))


| Group         | Setting     | Seed 1 | Seed 2 | Seed 3 | Total      |
| ------------- | ----------- | ------ | ------ | ------ | ---------- |
| GRPO Baseline | N=8, K=16   | 18.3   | 17.1   | 18.9   | 18.1 ± 0.9 |
| GRPO Baseline | N=4, K=32   | 16.0   | 20.6   | 19.4   | 18.7 ± 2.4 |
| GRPO Baseline | N=2, K=64   | 14.3   | 18.3   | 18.9   | 17.1 ± 2.5 |
| NGU           | NGU p=0.5   | 14.3   | 19.4   | 21.1   | 18.3 ± 3.6 |
| NGU           | NGU p=0.75  | 18.9   | 18.3   | 18.3   | 18.5 ± 0.3 |
| NGU           | NGU p=0.875 | 16.0   | 19.4   | 18.9   | 18.1 ± 1.8 |
